# Multi-Branch Retail Inventory Database

**Relational Database Design (AC51049) &mdash; analytical extension**
Yanyan Cai &middot; MSc Business Analytics and Financial Technology, University of Dundee

This notebook takes the eight-entity schema designed for the AC51049 coursework
(Supplier, Branch, Employee, Customer, Product, Inventory, Order, OrderDetails),
implements it with full constraint enforcement, loads a synthetic five-branch
dataset, and runs the analytical queries the model was designed to support.

### What is real and what is not

| Element | Status |
|---|---|
| The eight-entity schema and ER design | Original coursework, AC51049 |
| Constraint design (keys, foreign keys, CHECK) | Original |
| The eleven analytical queries | Written for this notebook |
| `StockCount` and `InventoryMovement` tables | Extensions proposed after the coursework |
| **All data** | **Synthetic and randomly generated. No real retailer's data appears here.** |

The synthetic data is generated with a fixed seed, so every figure below is
reproducible. Where a result matches a published industry benchmark, that is
because the generator was parameterised to sit in a realistic range. It is a
check that the query works, not a finding about real retail.

### Why DuckDB

DuckDB runs in-process with no server, so this notebook opens and runs in Google
Colab with a single `pip install`. It enforces primary keys, foreign keys and
CHECK constraints, and supports CTEs and window functions, which the queries use.

In [1]:
!pip install duckdb --quiet
import duckdb, random
import pandas as pd
from datetime import date, timedelta

random.seed(42)
pd.set_option('display.width', 200)
con = duckdb.connect()
print('DuckDB', duckdb.__version__)

DuckDB 1.5.5


## 1. Schema

Four design decisions are worth stating, because each blocks a category of bad
data that a spreadsheet estate cannot prevent.

1. **`Inventory` has a composite primary key `(ProductID, BranchID)`.** One
   product cannot hold two competing stock rows at the same branch. In a
   spreadsheet estate that duplication is the usual reason a group stock figure
   will not reconcile.
2. **Foreign keys on every reference.** Stock cannot be recorded against a
   product code or branch that does not exist. A mistyped code fails at write
   time rather than surfacing at year end.
3. **`CHECK (QuantityOnHand >= 0)`.** Negative stock is an accounting artefact,
   not a physical state, so it is refused.
4. **`Order` carries an `OrderType`.** The coursework model used a single order
   entity; separating sales from purchase orders explicitly is a refinement made
   here, so that supplier lead time and customer demand can both be analysed
   without ambiguity.

`ORDER` is a reserved word in SQL, so the table is quoted throughout.

In [2]:
SCHEMA = """
CREATE TABLE Supplier (
    SupplierID   INTEGER PRIMARY KEY,
    SupplierName VARCHAR NOT NULL,
    Country      VARCHAR NOT NULL,
    ContactEmail VARCHAR
);

CREATE TABLE Branch (
    BranchID   INTEGER PRIMARY KEY,
    BranchName VARCHAR NOT NULL,
    City       VARCHAR NOT NULL,
    OpenedDate DATE
);

CREATE TABLE Employee (
    EmployeeID INTEGER PRIMARY KEY,
    BranchID   INTEGER NOT NULL REFERENCES Branch(BranchID),
    FullName   VARCHAR NOT NULL,
    JobRole    VARCHAR NOT NULL,
    HiredDate  DATE
);

CREATE TABLE Customer (
    CustomerID   INTEGER PRIMARY KEY,
    CustomerName VARCHAR NOT NULL,
    City         VARCHAR,
    JoinedDate   DATE
);

CREATE TABLE Product (
    ProductID   INTEGER PRIMARY KEY,
    SKU         VARCHAR NOT NULL UNIQUE,
    ProductName VARCHAR NOT NULL,
    Category    VARCHAR NOT NULL,
    SupplierID  INTEGER NOT NULL REFERENCES Supplier(SupplierID),
    UnitCost    DECIMAL(10,2) NOT NULL CHECK (UnitCost  >= 0),
    UnitPrice   DECIMAL(10,2) NOT NULL CHECK (UnitPrice >= 0)
);

CREATE TABLE Inventory (
    ProductID      INTEGER NOT NULL REFERENCES Product(ProductID),
    BranchID       INTEGER NOT NULL REFERENCES Branch(BranchID),
    QuantityOnHand INTEGER NOT NULL DEFAULT 0 CHECK (QuantityOnHand >= 0),
    ReorderPoint   INTEGER NOT NULL DEFAULT 0,
    LastCounted    DATE,
    PRIMARY KEY (ProductID, BranchID)
);

CREATE TABLE "Order" (
    OrderID      INTEGER PRIMARY KEY,
    OrderType    VARCHAR NOT NULL CHECK (OrderType IN ('SALES','PURCHASE')),
    BranchID     INTEGER NOT NULL REFERENCES Branch(BranchID),
    EmployeeID   INTEGER          REFERENCES Employee(EmployeeID),
    CustomerID   INTEGER          REFERENCES Customer(CustomerID),
    SupplierID   INTEGER          REFERENCES Supplier(SupplierID),
    OrderDate    DATE NOT NULL,
    PromisedDate DATE,
    ReceivedDate DATE
);

CREATE TABLE OrderDetails (
    OrderID   INTEGER NOT NULL REFERENCES "Order"(OrderID),
    LineNo    INTEGER NOT NULL,
    ProductID INTEGER NOT NULL REFERENCES Product(ProductID),
    Quantity  INTEGER NOT NULL CHECK (Quantity > 0),
    UnitPrice DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (OrderID, LineNo)
);

CREATE TABLE StockCount (
    CountID    INTEGER PRIMARY KEY,
    ProductID  INTEGER NOT NULL REFERENCES Product(ProductID),
    BranchID   INTEGER NOT NULL REFERENCES Branch(BranchID),
    CountDate  DATE NOT NULL,
    CountedQty INTEGER NOT NULL CHECK (CountedQty >= 0),
    CountedBy  INTEGER REFERENCES Employee(EmployeeID)
);

CREATE TABLE InventoryMovement (
    MovementID   INTEGER PRIMARY KEY,
    ProductID    INTEGER NOT NULL REFERENCES Product(ProductID),
    BranchID     INTEGER NOT NULL REFERENCES Branch(BranchID),
    EmployeeID   INTEGER          REFERENCES Employee(EmployeeID),
    MovementType VARCHAR NOT NULL CHECK (MovementType IN
                 ('RECEIPT','SALE','TRANSFER_IN','TRANSFER_OUT',
                  'ADJUSTMENT','WASTE','RETURN')),
    Quantity     INTEGER NOT NULL,
    MovementDate DATE NOT NULL,
    SourceRef    VARCHAR
);
"""

for stmt in [s for s in SCHEMA.split(";") if s.strip()]:
    con.execute(stmt)
print("schema created")

schema created


## 2. Synthetic data

Five branches, six suppliers, 120 products, 80 customers, roughly 1,160 orders.

Three generator choices matter for the analysis that follows:

- **Demand follows a power law**, so ABC classification has something real to
  find rather than a flat distribution.
- **Suppliers are given different lead-time means and variances**, so the
  reliability query can separate a slow-but-consistent supplier from a
  fast-but-erratic one. Variance, not the average, is what drives safety stock.
- **60% of stock-count lines are seeded with a discrepancy**, chosen to sit near
  the 59.54% figure reported by the ECR Retail Loss field study across seven
  European retailers and 100 stores.

In [3]:
TODAY = date(2026, 9, 1)

sups = [(1,'Kelvin Wholesale Ltd','UK'),(2,'Tayside Provisions','UK'),
        (3,'Northgate Supply Co','UK'),(4,'Vantage Import Group','Netherlands'),
        (5,'Clyde Trading Partners','UK'),(6,'Meridian Sourcing BV','Belgium')]
con.executemany("INSERT INTO Supplier VALUES (?,?,?,?)",
    [(i,n,c,f"orders@{n.split()[0].lower()}.example") for i,n,c in sups])

brs = [(1,'Dundee Central','Dundee','2016-03-01'),(2,'Perth Road','Dundee','2018-06-15'),
       (3,'Arbroath','Arbroath','2019-11-04'),(4,'Perth High Street','Perth','2021-02-20'),
       (5,'St Andrews','St Andrews','2023-08-09')]
con.executemany("INSERT INTO Branch VALUES (?,?,?,?)", brs)

roles = ['Store Manager','Stock Controller','Sales Assistant']
emps = [(100+i, (i % 5)+1, f"Employee {i:02d}", roles[i % 3],
         str(date(2019,1,1)+timedelta(days=i*37))) for i in range(1,16)]
con.executemany("INSERT INTO Employee VALUES (?,?,?,?,?)", emps)

custs = [(1000+i, f"Customer {i:03d}", random.choice([b[2] for b in brs]),
          str(TODAY - timedelta(days=random.randint(30,1400)))) for i in range(1,81)]
con.executemany("INSERT INTO Customer VALUES (?,?,?,?)", custs)

cats = ['Ambient Grocery','Chilled','Household','Beverages','Health & Beauty']
prods = []
for i in range(1,121):
    cost = round(random.uniform(0.60, 24.00), 2)
    prods.append((i, f"SKU-{10000+i}", f"{random.choice(cats).split()[0]} Item {i:03d}",
                  random.choice(cats), random.randint(1,6), cost,
                  round(cost * random.uniform(1.25, 1.95), 2)))
con.executemany("INSERT INTO Product VALUES (?,?,?,?,?,?,?)", prods)

inv = []
for p in prods:
    for b in brs:
        rp = random.randint(8, 40)
        qty = max(0, int(rp * random.choice([0.2,0.5,0.8,1.2,1.6,2.4,3.2])))
        inv.append((p[0], b[0], qty, rp,
                    str(TODAY - timedelta(days=random.randint(5,120)))))
con.executemany("INSERT INTO Inventory VALUES (?,?,?,?,?)", inv)

# Pareto-shaped demand so ABC classification produces a realistic curve
weights = [1.0/(r**0.85) for r in range(1, 121)]
orders, lines, oid, lid = [], [], 1, 0
for _ in range(900):
    od = TODAY - timedelta(days=random.randint(1, 360))
    orders.append((oid,'SALES',random.randint(1,5),random.choice(emps)[0],
                   random.choice(custs)[0],None,str(od),None,None))
    for ln in range(1, random.randint(1,5)+1):
        p = random.choices(prods, weights=weights, k=1)[0]
        lines.append((oid, ln, p[0], random.randint(1,9), p[6]))
    oid += 1
for _ in range(260):
    od = TODAY - timedelta(days=random.randint(5, 360))
    sup = random.randint(1,6)
    promised = od + timedelta(days=[5,6,7,9,12,14][sup-1])
    slip = random.gauss([0.2,1.4,-0.3,3.1,0.8,2.6][sup-1], [1.0,2.2,0.8,4.0,1.5,3.4][sup-1])
    received = promised + timedelta(days=int(round(slip)))
    orders.append((oid,'PURCHASE',random.randint(1,5),random.choice(emps)[0],
                   None,sup,str(od),str(promised),str(received)))
    for ln in range(1, random.randint(2,6)+1):
        p = random.choice([x for x in prods if x[4]==sup] or prods)
        lines.append((oid, ln, p[0], random.randint(10,60), p[5]))
    oid += 1
con.executemany('INSERT INTO "Order" VALUES (?,?,?,?,?,?,?,?,?)', orders)
con.executemany("INSERT INTO OrderDetails VALUES (?,?,?,?,?)", lines)

# Stock counts: ~60% of counted lines carry a discrepancy, echoing the ECR finding
cnt, cid = [], 1
for p, b, qty, rp, lc in random.sample(inv, 300):
    if random.random() < 0.60:
        counted = max(0, qty + random.choice([-4,-3,-2,-1,1,2,3,5]))
    else:
        counted = qty
    cnt.append((cid, p, b, str(TODAY - timedelta(days=7)), counted, random.choice(emps)[0]))
    cid += 1
con.executemany("INSERT INTO StockCount VALUES (?,?,?,?,?,?)", cnt)

# Movement mix reflects real proportions: sales dominate, receipts replenish,
# and waste/adjustment are the small residual that a position-only table hides.
MV_TYPES  = ['SALE','RECEIPT','ADJUSTMENT','WASTE','RETURN']
MV_WEIGHT = [0.55,  0.25,     0.08,        0.07,   0.05]
MV_QTY = {'SALE':      lambda: -random.randint(1, 12),
          'RECEIPT':   lambda:  random.randint(20, 80),
          'ADJUSTMENT':lambda:  random.choice([-1,-2,-3,-4,-5, 1, 2, 3]),
          'WASTE':     lambda: -random.randint(1, 6),
          'RETURN':    lambda:  random.randint(1, 4)}
mv, mid = [], 1
for p, b, qty, rp, lc in random.sample(inv, 400):
    for _ in range(random.randint(1,4)):
        mt = random.choices(MV_TYPES, weights=MV_WEIGHT, k=1)[0]
        mv.append((mid, p, b, random.choice(emps)[0], mt, MV_QTY[mt](),
                   str(TODAY - timedelta(days=random.randint(1,180))), f"REF-{mid:05d}"))
        mid += 1
con.executemany("INSERT INTO InventoryMovement VALUES (?,?,?,?,?,?,?,?)", mv)

In [4]:
for t in ['Supplier','Branch','Employee','Customer','Product','Inventory',
          '"Order"','OrderDetails','StockCount','InventoryMovement']:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:22s} {n:>6,}")

Supplier                    6
Branch                      5
Employee                   15
Customer                   80
Product                   120
Inventory                 600
"Order"                 1,160
OrderDetails            3,708
StockCount                300
InventoryMovement       1,002


## 3. Constraints actually hold

A schema is only worth the constraints it enforces. Each statement below is a
realistic data-entry error, and each one is refused rather than stored.

In [5]:
bad = [
    ("stock for a product that does not exist",
     "INSERT INTO Inventory VALUES (99999, 1, 10, 5, DATE '2026-09-01')"),
    ("negative stock",
     "INSERT INTO Inventory VALUES (7, 2, -3, 5, DATE '2026-09-01')"),
    ("duplicate product/branch stock row",
     "INSERT INTO Inventory VALUES (1, 1, 10, 5, DATE '2026-09-01')"),
    ("order line for an order that does not exist",
     "INSERT INTO OrderDetails VALUES (999999, 1, 5, 2, 9.99)"),
    ("invalid movement type",
     "INSERT INTO InventoryMovement VALUES (99999,1,1,101,'BORROWED',5,DATE '2026-09-01','X')"),
]
for label, stmt in bad:
    try:
        con.execute(stmt)
        print(f"  NOT BLOCKED  {label}")
    except Exception as e:
        print(f"  blocked      {label}  ->  {type(e).__name__}")

  blocked      stock for a product that does not exist  ->  ConstraintException
  blocked      negative stock  ->  ConstraintException
  blocked      duplicate product/branch stock row  ->  ConstraintException
  blocked      order line for an order that does not exist  ->  ConstraintException
  blocked      invalid movement type  ->  ConstraintException


## 4. Analysis

Ten queries, ordered roughly by how quickly each pays for itself. Every one of
them requires a single source of product identity, which is precisely what five
separate spreadsheets cannot provide.

**Q1. Stock position for one product across all five branches.** The question a spreadsheet estate cannot answer without opening five files and trusting that all five use the same product code. `Headroom` is negative wherever the branch is below its reorder point.

In [6]:
q = """
SELECT  b.BranchName, i.QuantityOnHand, i.ReorderPoint,
        i.QuantityOnHand - i.ReorderPoint AS Headroom
FROM    Inventory i
JOIN    Branch  b ON b.BranchID  = i.BranchID
JOIN    Product p ON p.ProductID = i.ProductID
WHERE   p.SKU = 'SKU-10042'
ORDER BY Headroom;"""
con.execute(q).df()

,BranchName,QuantityOnHand,ReorderPoint,Headroom
0,Dundee Central,7,36,-29
1,Arbroath,3,17,-14
2,Perth Road,10,13,-3
3,St Andrews,33,28,5
4,Perth High Street,40,25,15


**Q2. Reorder-point breaches across the estate.** The daily operational list. Ranking by shortfall rather than alphabetically is what makes it actionable.

In [7]:
q = """
SELECT  b.BranchName, p.SKU, p.ProductName,
        i.QuantityOnHand, i.ReorderPoint,
        i.ReorderPoint - i.QuantityOnHand AS Shortfall
FROM    Inventory i
JOIN    Branch  b ON b.BranchID  = i.BranchID
JOIN    Product p ON p.ProductID = i.ProductID
WHERE   i.QuantityOnHand < i.ReorderPoint
ORDER BY Shortfall DESC
LIMIT 10;"""
con.execute(q).df()

,BranchName,SKU,ProductName,QuantityOnHand,ReorderPoint,Shortfall
0,Perth High Street,SKU-10002,Household Item 002,7,39,32
1,Arbroath,SKU-10011,Ambient Item 011,7,39,32
2,Arbroath,SKU-10014,Ambient Item 014,7,39,32
3,Perth Road,SKU-10034,Household Item 034,7,39,32
4,St Andrews,SKU-10044,Chilled Item 044,7,38,31
5,St Andrews,SKU-10026,Health Item 026,7,38,31
6,Perth Road,SKU-10077,Health Item 077,7,38,31
7,Arbroath,SKU-10031,Household Item 031,7,37,30
8,St Andrews,SKU-10036,Health Item 036,7,37,30
9,Perth High Street,SKU-10054,Health Item 054,7,37,30


**Q3. ABC classification by revenue contribution.** A cumulative-sum window function over descending revenue gives the Pareto curve. This matters operationally because the ECR study found record-accuracy uplift concentrates in fast-moving A-class items, so this query is what directs counting effort.

In [8]:
q = """
WITH line_value AS (
    SELECT od.ProductID, SUM(od.Quantity * od.UnitPrice) AS Revenue
    FROM   OrderDetails od
    JOIN   "Order" o ON o.OrderID = od.OrderID
    WHERE  o.OrderType = 'SALES'
      AND  o.OrderDate >= DATE '2025-09-01'
    GROUP BY od.ProductID
),
ranked AS (
    SELECT ProductID, Revenue,
           SUM(Revenue) OVER (ORDER BY Revenue DESC
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
           / SUM(Revenue) OVER () AS CumulativeShare
    FROM   line_value
)
SELECT  p.SKU, p.ProductName, ROUND(r.Revenue,2) AS Revenue,
        ROUND(r.CumulativeShare*100,1) AS CumPct,
        CASE WHEN r.CumulativeShare <= 0.80 THEN 'A'
             WHEN r.CumulativeShare <= 0.95 THEN 'B'
             ELSE 'C' END AS ABCClass
FROM    ranked r JOIN Product p ON p.ProductID = r.ProductID
ORDER BY r.Revenue DESC
LIMIT 12;"""
con.execute(q).df()

,SKU,ProductName,Revenue,CumPct,ABCClass
0,SKU-10002,Household Item 002,28289.24,10.3,A
1,SKU-10001,Ambient Item 001,25181.44,19.4,A
2,SKU-10005,Chilled Item 005,19142.20,26.3,A
3,SKU-10003,Ambient Item 003,15735.91,32.0,A
4,SKU-10007,Beverages Item 007,11530.40,36.2,A
5,SKU-10010,Ambient Item 010,8017.84,39.1,A
6,SKU-10011,Ambient Item 011,7544.88,41.8,A
7,SKU-10006,Chilled Item 006,7371.00,44.5,A
8,SKU-10020,Chilled Item 020,5974.48,46.7,A
9,SKU-10013,Beverages Item 013,5904.22,48.8,A


**Q3b. The same classification summarised.** The A/B/C split of product count against revenue share.

In [9]:
q = """
WITH line_value AS (
    SELECT od.ProductID, SUM(od.Quantity*od.UnitPrice) AS Revenue
    FROM OrderDetails od JOIN "Order" o ON o.OrderID = od.OrderID
    WHERE o.OrderType='SALES' GROUP BY od.ProductID),
ranked AS (
    SELECT ProductID, Revenue,
           SUM(Revenue) OVER (ORDER BY Revenue DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
           / SUM(Revenue) OVER () AS CumulativeShare
    FROM line_value)
SELECT  CASE WHEN CumulativeShare<=0.80 THEN 'A'
             WHEN CumulativeShare<=0.95 THEN 'B' ELSE 'C' END AS ABCClass,
        COUNT(*) AS Products,
        ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS PctOfProducts,
        ROUND(SUM(Revenue),2) AS Revenue,
        ROUND(100.0*SUM(Revenue)/SUM(SUM(Revenue)) OVER (),1) AS PctOfRevenue
FROM ranked GROUP BY 1 ORDER BY 1;"""
con.execute(q).df()

,ABCClass,Products,PctOfProducts,Revenue,PctOfRevenue
0,A,40,33.3,220230.48,79.8
1,B,40,33.3,41351.12,15.0
2,C,40,33.3,14309.49,5.2


**Q4. Stock turn and days of cover by branch.** The working-capital view. Two branches holding identical stock value can sit in very different cash positions once velocity is accounted for.

In [10]:
q = """
WITH sold AS (
    SELECT od.ProductID, o.BranchID, SUM(od.Quantity) AS Units
    FROM   OrderDetails od
    JOIN   "Order" o ON o.OrderID = od.OrderID
    WHERE  o.OrderType = 'SALES'
      AND  o.OrderDate >= CURRENT_DATE - INTERVAL '12 months'
    GROUP BY od.ProductID, o.BranchID
)
SELECT  b.BranchName,
        ROUND(SUM(i.QuantityOnHand * p.UnitCost),2)                  AS StockValue,
        ROUND(SUM(COALESCE(s.Units,0) * p.UnitCost),2)               AS COGS_12m,
        ROUND(SUM(COALESCE(s.Units,0) * p.UnitCost)
              / NULLIF(SUM(i.QuantityOnHand * p.UnitCost),0), 2)     AS StockTurn,
        ROUND(365 * SUM(i.QuantityOnHand * p.UnitCost)
              / NULLIF(SUM(COALESCE(s.Units,0) * p.UnitCost),0), 0)  AS DaysOfCover
FROM    Inventory i
JOIN    Branch  b ON b.BranchID  = i.BranchID
JOIN    Product p ON p.ProductID = i.ProductID
LEFT JOIN sold s ON s.ProductID = i.ProductID AND s.BranchID = i.BranchID
GROUP BY b.BranchName
ORDER BY DaysOfCover DESC;"""
con.execute(q).df()

,BranchName,StockValue,COGS_12m,StockTurn,DaysOfCover
0,Arbroath,56978.68,27995.97,0.49,743.0
1,St Andrews,57135.51,30245.41,0.53,690.0
2,Perth Road,54815.73,34976.16,0.64,572.0
3,Perth High Street,55850.82,39217.71,0.70,520.0
4,Dundee Central,46616.25,37489.03,0.80,454.0


**Q5. Supplier lead time and reliability.** Average lead time is the headline, but `LeadDayVariability` is what actually drives the safety stock a branch has to hold. A supplier that is slow and predictable is easier to plan around than one that is fast on average and erratic in practice.

In [11]:
q = """
SELECT  s.SupplierName,
        COUNT(*)                                             AS Orders,
        ROUND(AVG(DATE_DIFF('day', o.OrderDate, o.ReceivedDate)),1) AS AvgLeadDays,
        ROUND(STDDEV_SAMP(DATE_DIFF('day', o.OrderDate, o.ReceivedDate)),1)
                                                             AS LeadDayVariability,
        ROUND(100.0*SUM(CASE WHEN o.ReceivedDate <= o.PromisedDate
                             THEN 1 ELSE 0 END)/COUNT(*),1)   AS OnTimePct
FROM    "Order" o
JOIN    Supplier s ON s.SupplierID = o.SupplierID
WHERE   o.OrderType = 'PURCHASE' AND o.ReceivedDate IS NOT NULL
GROUP BY s.SupplierName
HAVING  COUNT(*) >= 5
ORDER BY OnTimePct ASC, LeadDayVariability DESC;"""
con.execute(q).df()

,SupplierName,Orders,AvgLeadDays,LeadDayVariability,OnTimePct
0,Vantage Import Group,40,12.7,4.4,20.0
1,Meridian Sourcing BV,48,16.7,4.2,27.1
2,Tayside Provisions,43,7.7,2.5,32.6
3,Clyde Trading Partners,38,12.7,1.5,39.5
4,Kelvin Wholesale Ltd,39,5.0,1.1,71.8
5,Northgate Supply Co,52,6.6,0.7,92.3


**Q6. Inter-branch imbalance.** The query that turns five shops into one estate: products below reorder point at one site while another site holds more than double its reorder point. The answer is a transfer, not a purchase order.

In [12]:
q = """
SELECT  p.SKU, p.ProductName,
        sh.BranchName AS ShortAt,   sh.QuantityOnHand AS ShortQty,
        su.BranchName AS SurplusAt, su.QuantityOnHand AS SurplusQty
FROM    Product p
JOIN   (SELECT i.ProductID, b.BranchName, i.QuantityOnHand
        FROM Inventory i JOIN Branch b ON b.BranchID = i.BranchID
        WHERE i.QuantityOnHand < i.ReorderPoint) sh ON sh.ProductID = p.ProductID
JOIN   (SELECT i.ProductID, b.BranchName, i.QuantityOnHand
        FROM Inventory i JOIN Branch b ON b.BranchID = i.BranchID
        WHERE i.QuantityOnHand > i.ReorderPoint * 2) su ON su.ProductID = p.ProductID
ORDER BY su.QuantityOnHand - sh.QuantityOnHand DESC
LIMIT 10;"""
con.execute(q).df()

,SKU,ProductName,ShortAt,ShortQty,SurplusAt,SurplusQty
0,SKU-10105,Beverages Item 105,Dundee Central,14,Arbroath,128
1,SKU-10076,Health Item 076,Dundee Central,15,St Andrews,128
2,SKU-10055,Ambient Item 055,Arbroath,6,St Andrews,118
3,SKU-10026,Health Item 026,St Andrews,7,Arbroath,118
4,SKU-10076,Health Item 076,Perth Road,17,St Andrews,128
5,SKU-10022,Beverages Item 022,Perth High Street,13,Dundee Central,124
6,SKU-10028,Beverages Item 028,Perth High Street,15,Dundee Central,124
7,SKU-10085,Household Item 085,Dundee Central,6,St Andrews,115
8,SKU-10047,Household Item 047,Arbroath,6,St Andrews,112
9,SKU-10119,Chilled Item 119,Perth Road,9,St Andrews,115


**Q7. Stock count against system quantity.** This is how a discrepancy is measured. Note that a small absolute variance on a low-stock line can be a very large percentage variance, which is why both columns are returned.

In [13]:
q = """
SELECT  b.BranchName, p.SKU,
        i.QuantityOnHand AS SystemQty, c.CountedQty,
        c.CountedQty - i.QuantityOnHand AS Variance,
        ROUND(100.0*ABS(c.CountedQty - i.QuantityOnHand)
              / NULLIF(i.QuantityOnHand,0),1) AS VariancePct
FROM    StockCount c
JOIN    Inventory i ON i.ProductID = c.ProductID AND i.BranchID = c.BranchID
JOIN    Branch  b ON b.BranchID  = c.BranchID
JOIN    Product p ON p.ProductID = c.ProductID
WHERE   c.CountedQty <> i.QuantityOnHand
ORDER BY ABS(c.CountedQty - i.QuantityOnHand) DESC
LIMIT 10;"""
con.execute(q).df()

,BranchName,SKU,SystemQty,CountedQty,Variance,VariancePct
0,St Andrews,SKU-10016,51,56,5,9.8
1,Dundee Central,SKU-10027,51,56,5,9.8
2,Perth Road,SKU-10023,5,10,5,100.0
3,Perth Road,SKU-10030,51,56,5,9.8
4,Dundee Central,SKU-10034,2,7,5,250.0
5,Dundee Central,SKU-10014,5,10,5,100.0
6,Perth High Street,SKU-10036,12,17,5,41.7
7,Arbroath,SKU-10003,76,81,5,6.6
8,Dundee Central,SKU-10033,26,31,5,19.2
9,Perth High Street,SKU-10038,8,13,5,62.5


**Q7b. Estate-wide discrepancy rate.** The generator seeded 60% of counted lines with a discrepancy; the query recovers that rate. This is a validation that the measurement works, not a finding about real retail. For reference, the ECR Retail Loss study found 59.54% of audited SKUs carried a discrepancy across seven European retailers.

In [14]:
q = """
SELECT  COUNT(*) AS LinesCounted,
        SUM(CASE WHEN c.CountedQty <> i.QuantityOnHand THEN 1 ELSE 0 END) AS Discrepant,
        ROUND(100.0*SUM(CASE WHEN c.CountedQty <> i.QuantityOnHand
                             THEN 1 ELSE 0 END)/COUNT(*),2) AS DiscrepancyPct
FROM    StockCount c
JOIN    Inventory i ON i.ProductID = c.ProductID AND i.BranchID = c.BranchID;"""
con.execute(q).df()

,LinesCounted,Discrepant,DiscrepancyPct
0,300,188.0,62.67


**Q8. Branch-level demand.** Where the Customer entity pays off. Branches serve different demand profiles and estate-wide averages hide that.

In [15]:
q = """
SELECT  b.BranchName,
        COUNT(DISTINCT o.CustomerID) AS ActiveCustomers,
        COUNT(DISTINCT o.OrderID)    AS Orders,
        ROUND(SUM(od.Quantity*od.UnitPrice),2) AS Revenue,
        ROUND(SUM(od.Quantity*od.UnitPrice)
              / NULLIF(COUNT(DISTINCT o.OrderID),0),2) AS AvgOrderValue
FROM    "Order" o
JOIN    OrderDetails od ON od.OrderID = o.OrderID
JOIN    Branch b ON b.BranchID = o.BranchID
WHERE   o.OrderType = 'SALES'
GROUP BY b.BranchName
ORDER BY Revenue DESC;"""
con.execute(q).df()

,BranchName,ActiveCustomers,Orders,Revenue,AvgOrderValue
0,Perth High Street,75,201,63247.39,314.66
1,Dundee Central,73,193,61220.54,317.20
2,Perth Road,72,178,56473.55,317.27
3,St Andrews,69,166,49598.14,298.78
4,Arbroath,71,162,45351.47,279.95


**Q9. Movement attribution.** This is the query that justifies the `InventoryMovement` extension. A position-only table can tell you stock is wrong; only a movement log can tell you whether it went to waste, to an unexplained adjustment, or out of the door. `UnexplainedPctOfReceipts` combines waste and adjustment as a share of goods received, which is the shrink signal a stock controller would actually be held to.

In [16]:
q = """
SELECT  b.BranchName,
        SUM(CASE WHEN m.MovementType='RECEIPT'    THEN m.Quantity ELSE 0 END) AS Receipts,
        SUM(CASE WHEN m.MovementType='SALE'       THEN m.Quantity ELSE 0 END) AS Sales,
        SUM(CASE WHEN m.MovementType='RETURN'     THEN m.Quantity ELSE 0 END) AS Returns,
        SUM(CASE WHEN m.MovementType='WASTE'      THEN m.Quantity ELSE 0 END) AS Waste,
        SUM(CASE WHEN m.MovementType='ADJUSTMENT' THEN m.Quantity ELSE 0 END) AS Adjustments,
        ROUND(100.0*ABS(SUM(CASE WHEN m.MovementType IN ('WASTE','ADJUSTMENT')
                                 THEN m.Quantity ELSE 0 END))
              / NULLIF(SUM(CASE WHEN m.MovementType='RECEIPT'
                                THEN m.Quantity ELSE 0 END),0),1) AS UnexplainedPctOfReceipts
FROM    InventoryMovement m
JOIN    Branch b ON b.BranchID = m.BranchID
GROUP BY b.BranchName
ORDER BY UnexplainedPctOfReceipts DESC;"""
con.execute(q).df()

,BranchName,Receipts,Sales,Returns,Waste,Adjustments,UnexplainedPctOfReceipts
0,Perth Road,3011.0,-666.0,42.0,-83.0,-38.0,4.0
1,Perth High Street,2100.0,-642.0,24.0,-60.0,-20.0,3.8
2,Dundee Central,2629.0,-602.0,26.0,-67.0,-22.0,3.4
3,Arbroath,2164.0,-625.0,39.0,-46.0,-22.0,3.1
4,St Andrews,2377.0,-688.0,31.0,-47.0,-10.0,2.4


## 5. What this schema still does not do

Stating the limits is part of the design, not an apology for it.

- **No point-of-sale transaction layer.** Sales are modelled as orders. A real
  retail estate would carry a POS transaction subject area, which is the largest
  single omission against the ARTS Operational Data Model, the retail industry
  reference design maintained by the Object Management Group.
- **No price or promotion history.** `UnitPrice` is stored on the order line, so
  historical prices survive, but there is no promotion entity and therefore no
  way to attribute a demand spike to a price action.
- **No merchandise hierarchy.** `Category` is a flat text field where a real
  model would carry department, class and subclass.
- **`QuantityOnHand` is still editable.** The `InventoryMovement` table is the
  first half of the fix; the second half is deriving position from the movement
  log rather than storing it independently, so that the two can never disagree.
- **This is an OLTP schema.** It is normalised for transaction processing and
  deliberately unsuited to reporting at volume. The next build is a separate
  dimensional layer: a periodic snapshot fact table of stock position by product,
  branch and day, with conformed dimensions and type-2 slowly changing dimensions
  so that reclassifying a product mid-year does not silently rewrite last year's
  numbers.
- **No indexing or query-plan work.** At this data volume it would not be
  measurable, and claiming it would be dishonest.

### Sources referenced

- Rekik, Y., Syntetos, A. and Glock, C. (2020). *Measuring the Sales Impact of
  Improving Inventory Records.* ECR Retail Loss.
- Object Management Group, Retail Domain Technology Committee. *ARTS Operational
  Data Model*, version 7.3.